In [ ]:
import os
import glob
import math
import itertools
from typing import Dict, List, Tuple, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import timm
from monai.transforms import Compose, NormalizeIntensityd, Resized, CenterSpatialCropd

import matplotlib.pyplot as plt

# Paths
DATA_ROOT = "data"
TEST_CSV = os.path.join(DATA_ROOT, "valid.csv")
SERIES_CSV = os.path.join(DATA_ROOT, "train_series.csv") # Предполагаем наличие этого файла
SERIES_ROOT = os.path.join(DATA_ROOT, "train_series")
WORKING_DIR = "." if os.path.isdir(".") else os.getcwd()

OUT_PATH = os.path.join(WORKING_DIR, "submission.csv")
CHECKPOINT_PATHS = [
    ("models/0.9159_refined_predictions_perclass_v3_efficientnet_b3.ra2_in1k/backbone_0.9159_refined_predictions_perclass_v3_efficientnet_b3.ra2_in1k.torchscript", "models/0.9159_refined_predictions_perclass_v3_efficientnet_b3.ra2_in1k/head_0.9159_refined_predictions_perclass_v3_efficientnet_b3.ra2_in1k.torchscript"),
    ("models/experimental_backbone.torchscript", "models/experimental_head.torchscript"),
]

# Constants
LABEL_COLUMNS = [
    "ACL", 
    "MCL", 
    "Medial Meniscus", 
    "Lateral Meniscus",
    "Medial OA", 
    "Lateral OA", 
    "PF OA", 
    "Effusion",
    "Synovitis", 
    "Baker's", 
    "Contusion", 
    "Fracture",
]
NUM_CLASSES = len(LABEL_COLUMNS)
SLICE_SIZE = (256, 256)

SLICE_COUNT = 32
SLICE_FRACTIONS = np.arange(0.1, 0.9, 0.8/SLICE_COUNT).tolist()

# Настройки параллелизации
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 1
MAX_WORKERS = NUM_GPUS * 2

print(f"Available GPUs: {NUM_GPUS}")

# Загружаем модели для каждого GPU
def load_models_for_device(device_id: int):
    """Загружает модели на указанный GPU"""
    device = f"cuda:{device_id}"
    models = []
    for backbone_path, head_path in CHECKPOINT_PATHS:
        backbone = torch.jit.load(backbone_path).eval().half().to(device)
        head = torch.jit.load(head_path).eval().half().to(device)
        models.append((backbone, head))
    return models, device

# Создаем пул моделей для каждого GPU
gpu_models = {}
for gpu_id in range(NUM_GPUS):
    models, device = load_models_for_device(gpu_id)
    gpu_models[gpu_id] = {
        'models': models,
        'device': device
    }

normalize_transform = Compose([
    Resized(keys=["image"], spatial_size=(-1, 384, 384)),
    CenterSpatialCropd(keys=["image"], roi_size=(-1, *SLICE_SIZE)),
    NormalizeIntensityd(keys=["image"]),
])

def load_and_prepare_series(series_path: str, target_size: Tuple[int, int] = (256, 256)) -> torch.Tensor:
    """Load DICOM series, sort, preprocess and normalize."""
    files_raw = [
        f for f in os.listdir(series_path)
        if os.path.isfile(os.path.join(series_path, f))
    ]

    file_info = []
    for f in files_raw:
        full_path = os.path.join(series_path, f)
        ds = pydicom.dcmread(full_path, stop_before_pixels=True)
        inst_num = getattr(ds, 'InstanceNumber', 0)
        file_info.append((int(inst_num), full_path))

    file_info.sort(key=lambda x: x[0])
    dicom_files = [f[1] for f in file_info]

    positions = []
    n_files = len(dicom_files)
    if n_files > SLICE_COUNT:
        for frac in SLICE_FRACTIONS:
            pos = int(n_files * frac)
            pos = int(np.clip(pos, 0, n_files - 1))
            positions.append(pos)
    else:
        positions = [i for i in range(n_files)]
    
    slices_data = []
    
    for pos in positions:
        f = dicom_files[pos]
        ds = pydicom.dcmread(f)
        pixel_array = ds.pixel_array.astype(np.float32)
        slices_data.append(pixel_array)

    processed_tensors = []
    for sl in slices_data:
        t = torch.from_numpy(sl)
        processed_tensors.append(t)
        
    volume_tensor = torch.stack(processed_tensors, dim=0).unsqueeze(0)
    
    data_dict = {"image": volume_tensor}
    norm_dict = normalize_transform(data_dict)

    return norm_dict["image"].permute(1, 0, 2, 3)



def process_single_study(row_data: tuple, gpu_id: int) -> Tuple[str, List[np.ndarray]]:
    """
    Обрабатывает одно исследование и возвращает логиты от каждой модели отдельно
    """
    idx, row = row_data
    study_uid = str(row["StudyInstanceUID"])
    
    try:
        study_dir = os.path.join(SERIES_ROOT, study_uid)
        
        planes_paths = {
            "Axial": [],
            "Sagittal": [],
            "Coronal": []
        }
        
        study_series = train_series_df[train_series_df["StudyInstanceUID"] == study_uid]

        for _, ser_row in study_series.iterrows():
            ser_uid = str(ser_row["SeriesInstanceUID"])
            plane = str(ser_row["Anatomical_Plane"]).strip()
            
            ser_path = os.path.join(study_dir, ser_uid)
            if os.path.isdir(ser_path):
                planes_paths[plane].append(ser_path)
        
        planes_tensors = {
            "Axial": [],
            "Sagittal": [],
            "Coronal": []
        }

        device = gpu_models[gpu_id]['device']
        
        for plane, paths in planes_paths.items():
            for p in paths:
                try:
                    t = load_and_prepare_series(p)
                    if t.shape[0] > 0:
                        planes_tensors[plane].append(t.to(device).half())
                except Exception as e:
                    print(f"Error loading series {p}: {e}")

        axial_list = planes_tensors["Axial"]
        sagittal_list = planes_tensors["Sagittal"]
        coronal_list = planes_tensors["Coronal"]

        model_logits_lists = [[] for _ in range(len(CHECKPOINT_PATHS))]
          
        models = gpu_models[gpu_id]['models']

        with torch.no_grad():
            for model_idx, (backbone, head) in enumerate(models):
                features = []
                features_2 = []
                for ax_vol in axial_list:
                    ax_batch = torch.stack([ax_vol])
                    feats = backbone(ax_batch)
                    features.append(feats)

                    ax_batch_flip = torch.stack([ax_vol.flip(dims=[-1])])
                    feats2 = backbone(ax_batch_flip)
                    features_2.append(feats2)
                for sag_vol in sagittal_list:
                    sag_batch = torch.stack([sag_vol])
                    feats = backbone(sag_batch)
                    features.append(feats)
                    features_2.append(feats)
                for cor_vol in coronal_list:
                    cor_batch = torch.stack([cor_vol])
                    feats = backbone(cor_batch)
                    features.append(feats)

                    cor_batch_flip = torch.stack([cor_vol.flip(dims=[-1])])
                    feats2 = backbone(cor_batch_flip)
                    features_2.append(feats2)
            
                feats1 = torch.cat(features, dim=2)
                feats2 = torch.cat(features_2, dim=2)
                feats = torch.cat([feats1, feats2], dim=0)
                logits = head(feats)
                avg_logits = logits.mean(dim=0).cpu().numpy()
                model_logits_lists[model_idx].append(avg_logits)

        # Усредняем логиты внутри каждой модели по всем комбинациям серий
        model_avg_logits = []
        for model_idx in range(len(CHECKPOINT_PATHS)):
            if model_logits_lists[model_idx]:
                avg = np.mean(model_logits_lists[model_idx], axis=0)
                model_avg_logits.append(avg)
            else:
                model_avg_logits.append(np.zeros(NUM_CLASSES))
            
        return study_uid, model_avg_logits

    except Exception as e:
        print(f"Error processing study {study_uid}: {e}")
        return study_uid, [np.zeros(NUM_CLASSES)] * len(CHECKPOINT_PATHS)


# Загружаем данные
test_df = pd.read_csv(TEST_CSV, dtype={"StudyInstanceUID": str})
train_series_df = pd.read_csv(SERIES_CSV, dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str})

print(f"Total studies: {len(test_df)}")
print(f"Using {NUM_GPUS} GPUs")

# Собираем логиты от каждой модели отдельно
study_rows = list(test_df.iterrows())
tasks_per_gpu = [[] for _ in range(NUM_GPUS)]
for i, task in enumerate(study_rows):
    gpu_id = i % NUM_GPUS
    tasks_per_gpu[gpu_id].append(task)

# Структура для хранения логитов: {study_uid: [logits_model_0, logits_model_1, ...]}
all_model_logits = {}
lock = threading.Lock()

def worker_task_separate(task_batch: List[tuple], gpu_id: int) -> Dict[str, List[np.ndarray]]:
    """Обработчик для одного GPU с разделением логитов по моделям"""
    local_results = {}
    for task in tqdm(task_batch, desc=f"GPU {gpu_id}", position=gpu_id):
        uid, model_logits = process_single_study(task, gpu_id)
        local_results[uid] = model_logits
    return local_results

with ThreadPoolExecutor(max_workers=NUM_GPUS) as executor:
    futures = {}
    for gpu_id in range(NUM_GPUS):
        if tasks_per_gpu[gpu_id]:
            future = executor.submit(worker_task_separate, tasks_per_gpu[gpu_id], gpu_id)
            futures[future] = gpu_id
    
    for future in as_completed(futures):
        gpu_id = futures[future]
        try:
            gpu_results = future.result()
            with lock:
                all_model_logits.update(gpu_results)
        except Exception as e:
            print(f"Error in GPU {gpu_id} worker: {e}")

print(f"Processed {len(all_model_logits)} studies")





Available GPUs: 1
Total studies: 58
Using 1 GPUs


GPU 0:   0%|          | 0/58 [00:00<?, ?it/s]

Processed 58 studies


In [10]:
from itertools import product
from sklearn.metrics import roc_auc_score

print("\n" + "=" * 70)
print("GRID SEARCH ОПТИМИЗАЦИЯ ВЕСОВ (шаг 0.1)")
print("=" * 70)

def compute_weighted_predictions(weights: np.ndarray, model_logits_dict: Dict[str, List[np.ndarray]], 
                                  class_idx: int) -> Dict[str, float]:
    """Вычисляет взвешенные предсказания для конкретного класса"""
    predictions = {}
    
    for uid, model_logits_list in model_logits_dict.items():
        class_logits = np.array([logits[class_idx] for logits in model_logits_list])
        weighted_logit = np.sum(weights * class_logits)
        prob = 1 / (1 + np.exp(-weighted_logit))
        prob = np.clip(prob, 0, 1)
        predictions[uid] = prob
    
    return predictions

def generate_weight_combinations(num_models, step=0.1):
    """Генерирует все возможные комбинации весов с заданным шагом"""
    # Создаем список возможных значений весов
    values = np.arange(0.05, 1.0, step)
    
    # Генерируем все комбинации
    combinations = []
    
    if num_models == 2:
        for w1 in values:
            w2 = 1.0 - w1
            if w2 >= step:
                combinations.append(np.array([w1, w2]))
    
    elif num_models == 3:
        for w1 in values:
            for w2 in values:
                w3 = 1.0 - w1 - w2
                if w3 >= step:
                    combinations.append(np.array([w1, w2, w3]))
    
    elif num_models == 4:
        for w1 in values:
            for w2 in values:
                for w3 in values:
                    w4 = 1.0 - w1 - w2 - w3
                    if w4 >= step:
                        combinations.append(np.array([w1, w2, w3, w4]))
    
    else:
        # Для большего числа моделей используем рекурсивный подход
        def generate_recursive(remaining_models, remaining_weight, current_weights):
            if remaining_models == 1:
                if remaining_weight >= step:
                    combinations.append(np.array(current_weights + [remaining_weight]))
                return
            
            for w in values:
                if w <= remaining_weight - step * (remaining_models - 1):
                    generate_recursive(
                        remaining_models - 1, 
                        remaining_weight - w, 
                        current_weights + [w]
                    )
        
        generate_recursive(num_models, 1.0, [])
    
    print(f"  Сгенерировано {len(combinations)} комбинаций весов")
    return combinations


def evaluate_weights(weights, model_logits_dict, gt_df, class_idx):
    """Вычисляет ROC AUC для заданных весов"""
    predictions = compute_weighted_predictions(weights, model_logits_dict, class_idx)
    
    y_true_list = []
    y_pred_list = []
    
    for uid in gt_df["StudyInstanceUID"].values:
        mask = gt_df["StudyInstanceUID"] == uid
        y_true = gt_df.loc[mask, LABEL_COLUMNS[class_idx]].values[0]
        
        if not pd.isna(y_true) and uid in predictions:
            y_true_list.append(y_true)
            y_pred_list.append(predictions[uid])
    
    if len(y_true_list) < 2 or len(np.unique(y_true_list)) < 2:
        return np.nan
    
    try:
        auc = roc_auc_score(y_true_list, y_pred_list)
        return auc
    except:
        return np.nan


num_models = len(CHECKPOINT_PATHS)
optimal_weights_per_class = {}

# Генерируем комбинации весов
weight_combinations = generate_weight_combinations(num_models, step=0.05)

for class_idx, col_name in enumerate(LABEL_COLUMNS):
    print(f"\nПоиск для класса: {col_name}")
    print("-" * 70)
    
    best_auc = -float('inf')
    best_weights = np.ones(num_models) / num_models
    
    # Перебираем все комбинации
    for weights in tqdm(weight_combinations, desc=f"Класс {class_idx}", leave=False):
        auc = evaluate_weights(weights, all_model_logits, test_df, class_idx)
        
        if not np.isnan(auc) and auc > best_auc:
            best_auc = auc
            best_weights = weights.copy()
    
    optimal_weights_per_class[class_idx] = best_weights
    print(f"  ✓ Лучший результат: AUC={best_auc:.4f}, веса={best_weights.round(2)}")




GRID SEARCH ОПТИМИЗАЦИЯ ВЕСОВ (шаг 0.1)
  Сгенерировано 48691 комбинаций весов

Поиск для класса: ACL
----------------------------------------------------------------------


Класс 0:   0%|          | 0/48691 [00:00<?, ?it/s]

  ✓ Лучший результат: AUC=0.9914, веса=[0.05 0.05 0.05 0.05 0.35 0.05 0.05 0.05 0.05 0.05 0.2 ]

Поиск для класса: MCL
----------------------------------------------------------------------


Класс 1:   0%|          | 0/48691 [00:00<?, ?it/s]

  ✓ Лучший результат: AUC=1.0000, веса=[0.05 0.05 0.05 0.05 0.15 0.05 0.05 0.05 0.05 0.25 0.2 ]

Поиск для класса: Medial Meniscus
----------------------------------------------------------------------


Класс 2:   0%|          | 0/48691 [00:00<?, ?it/s]

... (15 outputs left out)

Класс 10:   0%|          | 0/48691 [00:00<?, ?it/s]

  ✓ Лучший результат: AUC=0.9447, веса=[0.05 0.05 0.05 0.25 0.05 0.05 0.05 0.05 0.05 0.25 0.1 ]

Поиск для класса: Fracture
----------------------------------------------------------------------


Класс 11:   0%|          | 0/48691 [00:00<?, ?it/s]

  ✓ Лучший результат: AUC=0.9514, веса=[0.05 0.05 0.05 0.05 0.25 0.05 0.05 0.1  0.05 0.2  0.1 ]


In [11]:
optimal_weights_per_class = {k:v.tolist() for k,v in optimal_weights_per_class.items()}
optimal_weights_per_class

{0: [0.05,
  0.05,
  0.05,
  0.05,
  0.35000000000000003,
  0.05,
  0.05,
  0.05,
  0.05,
  0.05,
  0.19999999999999984],
 1: [0.05,
  0.05,
  0.05,
  0.05,
  0.15000000000000002,
  0.05,
  0.05,
  0.05,
  0.05,
  0.25,
  0.19999999999999973],
 2: [0.05,
  0.05,
  0.05,
  0.05,
  0.05,
  0.05,
  0.05,
  0.05,
  0.4,
  0.1,
  0.09999999999999962],
 3: [0.05,
  0.05,
  0.35000000000000003,
  0.05,
  0.1,
  0.05,
  0.05,
  0.05,
  0.05,
  0.05,
  0.14999999999999986],
 4: [0.05,
  0.05,
  0.05,
  0.05,
  0.05,
  0.05,
  0.05,
  0.05,
  0.05,
  0.05,
  0.4999999999999996],
 5: [0.05,
  0.05,
  0.15000000000000002,
  0.05,
  0.05,
  0.05,
  0.05,
  0.3,
  0.05,
  0.1,
  0.09999999999999973],
 6: [0.2,
  0.05,
  0.05,
  0.1,
  0.05,
  0.05,
  0.05,
  0.05,
  0.05,
  0.15000000000000002,
  0.19999999999999996],
 7: [0.05,
  0.05,
  0.05,
  0.05,
  0.05,
  0.05,
  0.05,
  0.1,
  0.4,
  0.05,
  0.09999999999999969],
 8: [0.05,
  0.05,
  0.05,
  0.05,
  0.05,
  0.05,
  0.1,
  0.25,
  0.05,
  0.1

In [12]:
OPTIMAL_WEIGHTS = optimal_weights_per_class




def compute_final_predictions(weights_dict: Dict[int, np.ndarray], 
                              model_logits_dict: Dict[str, List[np.ndarray]]) -> pd.DataFrame:
    """Создает финальную submission с использованием заданных весов"""
    sub = test_df[["StudyInstanceUID"]].copy()
    
    for class_idx, col_name in enumerate(LABEL_COLUMNS):
        weights = weights_dict.get(class_idx, np.ones(len(CHECKPOINT_PATHS)) / len(CHECKPOINT_PATHS))
        weights = weights / np.sum(weights)  # Нормализуем
        
        probs = []
        for uid in sub["StudyInstanceUID"]:
            if uid in model_logits_dict:
                model_logits_list = model_logits_dict[uid]
                class_logits = np.array([logits[class_idx] for logits in model_logits_list])
                weighted_logit = np.sum(weights * class_logits)
                prob = 1 / (1 + np.exp(-weighted_logit))
                prob = np.clip(prob, 0, 1)
            else:
                prob = 0.5
            probs.append(prob)
        
        sub[col_name] = probs
    
    return sub


sub = compute_final_predictions(OPTIMAL_WEIGHTS, all_model_logits)

sub.to_csv(OUT_PATH, index=False)
print("Submission saved to:", OUT_PATH)
print(sub.head())



Submission saved to: ./submission.csv
                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10095687747295410396...  0.049369  0.076695   
1  1.2.826.0.1.3680043.8.498.10170898615867673028...  0.080222  0.114359   
2  1.2.826.0.1.3680043.8.498.10306159113324811538...  0.086707  0.083381   
3  1.2.826.0.1.3680043.8.498.11287937729196958426...  0.076048  0.110130   
4  1.2.826.0.1.3680043.8.498.11382021393803389951...  0.983620  0.125959   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.065363          0.293519   0.208338    0.281211  0.439043   
1         0.068042          0.377057   0.230075    0.390647  0.714708   
2         0.805846          0.280132   0.909594    0.526445  0.922994   
3         0.110329          0.200024   0.273077    0.211601  0.423325   
4         0.871015          0.376181   0.129362    0.107921  0.069722   

   Effusion  Synovitis   Baker's  Contusion  Fracture  
0  0.53952

In [13]:
# Валидация остается без изменений
from sklearn.metrics import roc_auc_score

df_gt_labels = test_df[LABEL_COLUMNS]
df_pred_labels = sub[LABEL_COLUMNS]

print("\n" + "=" * 70)
print("ВАЛИДАЦИЯ")
print("=" * 70)

roc_auc_results = {}
for col in LABEL_COLUMNS:
    mask = df_gt_labels[col].notna() & df_pred_labels[col].notna()
    if mask.sum() > 0:
        y_true = df_gt_labels.loc[mask, col].values
        y_scores = df_pred_labels.loc[mask, col].values
        
        if len(np.unique(y_true)) > 1:
            try:
                auc_score = roc_auc_score(y_true, y_scores)
                roc_auc_results[col] = auc_score
            except:
                roc_auc_results[col] = np.nan
        else:
            roc_auc_results[col] = np.nan
    else:
        roc_auc_results[col] = np.nan

roc_auc_series = pd.Series(roc_auc_results, name='ROC AUC')
print(roc_auc_series.round(4))
print(f"\nСредний ROC AUC: {np.nanmean(list(roc_auc_results.values())):.4f}")


ВАЛИДАЦИЯ
ACL                 0.9914
MCL                 1.0000
Medial Meniscus     0.9820
Lateral Meniscus    0.9280
Medial OA           0.9876
Lateral OA          0.8956
PF OA               0.9125
Effusion            0.9938
Synovitis           0.8805
Baker's             0.9982
Contusion           0.9447
Fracture            0.9514
Name: ROC AUC, dtype: float64

Средний ROC AUC: 0.9555
